# Ablation Run

This notebook runs the ablation CV using a **robust baseline pipeline** (imputation + OneHot for categoricals + RandomForest).
It exports `ablation_results.csv`, which is the input for `ablation_inference.ipynb`.


In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor


import sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.ablation_study import run_ablation_cv


In [ ]:
# -----------------------
# Configuration
# -----------------------
TRAIN_CSV = os.path.join(PROJECT_ROOT, "Data", "train.csv")  
TARGET_COL = "price"
ID_COL = "carID"

CV_FOLDS = 5
SEED = 42

OUT_RESULTS = os.path.join(PROJECT_ROOT, "ablation_results.csv")


In [5]:
# -----------------------
# Load train
# -----------------------
train_df = pd.read_csv(TRAIN_CSV)

if TARGET_COL not in train_df.columns:
    raise ValueError(f"Expected target column '{TARGET_COL}' in {TRAIN_CSV}")

X_full = train_df.drop(columns=[TARGET_COL, ID_COL], errors="ignore")
y = train_df[TARGET_COL]

print("X_full shape:", X_full.shape)
print("y shape:", y.shape)
print("Columns:", list(X_full.columns))


X_full shape: (75973, 12)
y shape: (75973,)
Columns: ['Brand', 'model', 'year', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage']


In [ ]:
# -----------------------
# Baseline pipeline builder (RandomForest + OneHot)
# -----------------------

def build_pipeline(use_cols):
    # Infer categorical vs numerical columns from the raw training dataframe
    df = train_df[use_cols]
    cat_cols = [c for c in use_cols if df[c].dtype == "object"]
    num_cols = [c for c in use_cols if c not in cat_cols]

    numeric = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])

    categorical = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", numeric, num_cols),
            ("cat", categorical, cat_cols),
        ],
        remainder="drop",
    )

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    return Pipeline(steps=[("preprocess", pre), ("model", model)])


In [7]:
# -----------------------
# Run ablation CV
# -----------------------
ablation_df = run_ablation_cv(
    X_full, y,
    builder=build_pipeline,
    cv_folds=CV_FOLDS,
    random_state=SEED,
    baseline_name="Baseline"
)

print("Ablation results shape:", ablation_df.shape)
ablation_df.head()


[ablation_cv] fold 1/5 done (7.1s elapsed)
[ablation_cv] fold 2/5 done (13.1s elapsed)
[ablation_cv] fold 3/5 done (19.5s elapsed)
[ablation_cv] fold 4/5 done (25.5s elapsed)
[ablation_cv] fold 5/5 done (31.4s elapsed)
Ablation results shape: (65, 9)


,model_variant,features_removed,fold,r2,adjusted_r2,rmse,mae,mse,mape
0,Baseline,none,1,0.822862,0.822722,4016.701630,2539.674175,1.613389e+07,19.210822
1,drop_Brand,Brand,1,0.821474,0.821344,4032.417373,2544.295243,1.626039e+07,19.228063
2,drop_model,model,1,0.728979,0.728783,4968.391367,3191.200965,2.468491e+07,23.668359
3,drop_year,year,1,0.787950,0.787796,4394.741854,2877.788683,1.931376e+07,20.625859
4,drop_transmission,transmission,1,0.816655,0.816523,4086.469915,2602.882688,1.669924e+07,19.786325


In [8]:
# -----------------------
# Save results (input for Person 2 notebook)
# -----------------------
ablation_df.to_csv(OUT_RESULTS, index=False)
print("Saved:", OUT_RESULTS)

# Sanity check: required columns
required_cols = {"model_variant", "features_removed", "fold", "r2", "adjusted_r2", "rmse", "mae", "mse", "mape"}
missing = required_cols - set(ablation_df.columns)
print("Missing columns:", missing)


Saved: c:\Users\rodri\Downloads\ML_group_45-main\ablation_results.csv
Missing columns: set()


## Next
Run `ablation_inference.ipynb` with:

- `INPUT_CSV = "ablation_results.csv"`

It should produce:
- `ablation_summary_stats.csv`
- `ablation_significance_tests.csv`
- `ablation_feature_importance.csv`
- `ablation_deploy_check.csv`
